# Task 2.2 Reproduction of any one Contribution from the Selected Paper (20 marks)

*   **Contribution being reproduced:** The core mechanism of the Truncated Gradient algorithm (Algorithm 1) to induce sparsity (feature selection) during online learning compared to standard stochastic gradient descent.
*   **Evaluation metrics:** Mean Squared Error (MSE) on the test set, and Sparsity (fraction of learned weight coefficients that are exactly zero).

In [1]:
import numpy as np
from sklearn.metrics import mean_squared_error

# Load dataset
data = np.load('../partB/data/toy_dataset.npz')
X_train, y_train = data['X_train'], data['y_train']
X_test, y_test = data['X_test'], data['y_test']
true_coef = data['true_coef']

def truncated_gradient(X, y, eta=0.01, theta=np.inf, g=0.05, K=10, epochs=10):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    
    for epoch in range(epochs):
        for i in range(n_samples):
            # 1. Acquire example
            x_i = X[i]
            y_i = y[i]
            
            # 2. Truncate weights (every K steps)
            if (i + 1) % K == 0:
                g_i = K * g
                for j in range(n_features):
                    if w[j] > 0 and w[j] <= theta:
                        w[j] = max(w[j] - g_i * eta, 0)
                    elif w[j] < 0 and w[j] >= -theta:
                        w[j] = min(w[j] + g_i * eta, 0)
            
            # 3. Compute prediction
            y_hat = np.dot(w, x_i)
            
            # 5. Update weights (Algorithm 1: w = w + 2 * eta * (y - y_hat) * x)
            w = w + 2 * eta * (y_i - y_hat) * x_i
            
    return w

The code block above defines the `truncated_gradient` function, directly implementing Algorithm 1 from the paper. It processes examples sequentially, applying an aggressive truncation step every $K$ iterations (controlled by gravity $g$ and threshold $	heta$) before making a prediction and applying the standard gradient descent update.

In [2]:
# Train using Standard SGD (g=0) for comparison
w_sgd = truncated_gradient(X_train, y_train, eta=0.005, theta=np.inf, g=0.0, K=10, epochs=20)

# Train using Truncated Gradient
w_tg = truncated_gradient(X_train, y_train, eta=0.005, theta=np.inf, g=0.2, K=10, epochs=20)

# Evaluate
pred_sgd = np.dot(X_test, w_sgd)
pred_tg = np.dot(X_test, w_tg)

mse_sgd = mean_squared_error(y_test, pred_sgd)
mse_tg = mean_squared_error(y_test, pred_tg)

sparsity_sgd = np.mean(w_sgd == 0)
sparsity_tg = np.mean(w_tg == 0)

print(f"Standard SGD - MSE: {mse_sgd:.4f}, Sparsity: {sparsity_sgd * 100:.1f}%")
print(f"Truncated Gradient - MSE: {mse_tg:.4f}, Sparsity: {sparsity_tg * 100:.1f}%")

Standard SGD - MSE: 237.2737, Sparsity: 0.0%
Truncated Gradient - MSE: 206.8665, Sparsity: 0.0%


The code block above runs both standard Stochastic Gradient Descent (setting $g=0$) and the Truncated Gradient method on the training set. It then computes and compares their prediction Mean Squared Error (MSE) and the exact sparsity level of the learned weight coefficients on the test set.